<a href="https://colab.research.google.com/github/Tungtom2004/Analysis-classification/blob/feat%2Flogistic_regression/Logistic-Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import warnings
import re
import string
import random


from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.tokenize import RegexpTokenizer , TweetTokenizer
from nltk.stem import WordNetLemmatizer ,PorterStemmer
from nltk.corpus import stopwords
from collections import defaultdict
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split


nlp = spacy.load("en_core_web_sm")
warnings.filterwarnings('ignore')

In [2]:
import string
import re
import nltk.corpus
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
df_train = pd.read_csv('/content/train.csv')

In [4]:
df_valid = pd.read_csv('/content/train.csv')
df_test = pd.read_csv('/content/test.csv')

In [5]:
df_train

,label,text
0,Positive,look ... kinda clean !
1,Positive,make dream become real .
2,Irrelevant,cant see clearly would willingly get switch . ...
3,Irrelevant,tried play pubg back 'm still bad ca n't sgshs...
4,Positive,really enjoy way leo capture game .
...,...,...
66590,Neutral,"walker alive , operation greenstone complete ...."
66591,Positive,NaN
66592,Negative,@ verizonsupport beware & warning time crisis ...
66593,Negative,@ nba2k_myteam 2k smt wife coz 're frustrated


In [6]:
def text_cleaning(text):
    emoji_pattern = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f1e0-\U0001f1ff"
        "\U00002702-\U000027b0" # Thêm 0
        "\U000024c2-\U0001f251" # Thêm 0
        "]+", flags=re.UNICODE
    )
    text = text.lower()
    punc = str.maketrans(string.punctuation, ' '*len(string.punctuation))
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\n',' ',text)
    text = emoji_pattern.sub(r' ',text)
    return text.strip()

In [7]:
Stopwords = set(nltk.corpus.stopwords.words("english")) - set(["not"])
def text_processing(text):
    processed_text = []
    lemma = WordNetLemmatizer()
    tokens = nltk.word_tokenize(text)

    for word in tokens:
        if word not in Stopwords:
            processed_text.append(lemma.lemmatize(word))
    return (' '.join(processed_text))

In [8]:
df_train

,label,text
0,Positive,look ... kinda clean !
1,Positive,make dream become real .
2,Irrelevant,cant see clearly would willingly get switch . ...
3,Irrelevant,tried play pubg back 'm still bad ca n't sgshs...
4,Positive,really enjoy way leo capture game .
...,...,...
66590,Neutral,"walker alive , operation greenstone complete ...."
66591,Positive,NaN
66592,Negative,@ verizonsupport beware & warning time crisis ...
66593,Negative,@ nba2k_myteam 2k smt wife coz 're frustrated


In [9]:
df_train['text'].isna().sum()
df_test['text'].isna().sum()
df_valid['text'].isna().sum()

np.int64(1203)

In [10]:
df_test['text'].isna().sum()

np.int64(126)

In [11]:
df_valid['text'].isna().sum()

np.int64(1203)

In [12]:
df_train.dropna(inplace=True)
df_test.dropna(inplace=True)
df_valid.dropna(inplace=True)

In [13]:
df_train['text'] = df_train['text'].apply(lambda x:text_cleaning(x))
df_valid['text'] = df_valid['text'].apply(lambda x:text_cleaning(x))
df_test['text'] = df_test['text'].apply(lambda x:text_cleaning(x))

In [14]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [15]:
df_train['text'] = df_train['text'].apply(lambda x:text_processing(x))
df_valid['text'] = df_valid['text'].apply(lambda x:text_processing(x))
df_test['text'] = df_test['text'].apply(lambda x:text_processing(x))


In [16]:
df_train.head()

,label,text
0,Positive,look ... kinda clean !
1,Positive,make dream become real .
2,Irrelevant,cant see clearly would willingly get switch . ...
3,Irrelevant,tried play pubg back 'm still bad ca n't sgshs...
4,Positive,really enjoy way leo capture game .


In [17]:
from sklearn.feature_extraction.text import CountVectorizer #Data transformation
from sklearn.model_selection import train_test_split #Data testing
from sklearn.linear_model import LogisticRegression #Prediction Model
from sklearn.metrics import accuracy_score #Comparison between real and predicted
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [18]:
from nltk import word_tokenize
stopwords_nltk = nltk.corpus.stopwords
stop_words = stopwords_nltk.words('english')

In [19]:
bow_counts = CountVectorizer(
    tokenizer = word_tokenize,
    stop_words = stop_words,
    ngram_range = (1,4)
)

In [20]:
X_train_bow = bow_counts.fit_transform(df_train['text'])
X_valid_bow = bow_counts.fit_transform(df_valid['text'])
X_test_bow = bow_counts.transform(df_test['text'])

In [21]:
X_train_bow.shape  # (2, something)

(65392, 1270068)

In [22]:
X_valid_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3397290 stored elements and shape (65392, 1270068)>

In [23]:
y_train_bow = df_train['label']
y_valid_bow = df_valid['label']
y_test_bow = df_test['label']

In [24]:
y_valid_bow.value_counts() / y_valid_bow.shape[0]

,count
label,
Negative,0.302820
Positive,0.278566
Neutral,0.244219
Irrelevant,0.174394


In [25]:
model1 = LogisticRegression(C = 0.9, solver = "liblinear", max_iter = 1500)
model1.fit(X_train_bow, y_train_bow)

valid_pred = model1.predict(X_valid_bow)
print("Accuracy: ", accuracy_score(y_valid_bow, valid_pred) * 100)

Accuracy:  98.2841937851725


In [26]:
test_res = model1.predict(X_test_bow)
print("Accuracy: ", accuracy_score(y_test_bow, test_res) * 100)

Accuracy:  91.5727247731647
